In [ ]:
import os
import glob
import pandas as pd

# ---------------------------------------------------------
# 1. 环境依赖检查 (按需导入空间分析库)
# ---------------------------------------------------------
try:
    import rasterio
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print("⚠️ 提示：未检测到 rasterio 库。如果需要读取 .tif 栅格文件，请先运行 !pip install rasterio")

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False
    print("⚠️ 提示：未检测到 geopandas 库。如果需要读取 .shp 矢量文件，请先运行 !pip install geopandas")

# ---------------------------------------------------------
# 2. 定义文件路径与数据存储字典
# ---------------------------------------------------------
base_dir = r"E:\chuli"
subfolders = [
    'NY_independent_variable',  # 纽约：自变量 (如街景视觉、POI可达性等)
    'HK_independent_variable',  # 香港：自变量
    'NY_dependent_variable',    # 纽约：因变量 (五类犯罪频次等)
    'HK_dependent_variable'     # 香港：因变量 (交通事故频次等)
]

# 用于在内存中保存读取的数据，方便你后续直接调用
tabular_data = {}     # 存储所有的 DataFrame
raster_metadata = {}  # 存储栅格文件的元数据(不全量读入内存以防溢出)
vector_metadata = {}  # 存储矢量文件的 GeoDataFrame 或元数据

print("\n================ 开始全类型空间与属性数据扫描 ================\n")

# ---------------------------------------------------------
# 3. 遍历文件夹并分类读取
# ---------------------------------------------------------
for folder in subfolders:
    folder_path = os.path.join(base_dir, folder)
    
    if not os.path.exists(folder_path):
        print(f"❌ 未找到文件夹: {folder_path}，请检查硬盘或路径拼写。")
        continue
        
    print(f"📁 正在深度扫描: 【{folder}】 ...")
    
    # 支持的文件格式后缀
    patterns = ["*.csv", "*.xlsx", "*.xls", "*.tif", "*.tiff", "*.shp"]
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(folder_path, p)))
        
    if not files:
        print(f"  -> 提示：在该文件夹中未找到支持的表格或空间文件。\n")
        continue
        
    for file in files:
        file_name = os.path.basename(file)
        file_ext = os.path.splitext(file_name)[1].lower()
        file_key = f"{folder}_{file_name}"  # 设定唯一键名，防止重名
        
        try:
            # --- 类别 A: 读取表格数据 (CSV/Excel) ---
            if file_ext in ['.csv', '.xlsx', '.xls']:
                if file_ext == '.csv':
                    # 尝试处理可能存在的编码问题
                    try:
                        df = pd.read_csv(file)
                    except UnicodeDecodeError:
                        df = pd.read_csv(file, encoding='gbk')
                else:
                    df = pd.read_excel(file)
                    
                tabular_data[file_key] = df
                print(f"  📄 [表格] {file_name}")
                print(f"     -> 维度 (行x列): {df.shape}")
                
                # 展示前5个列名，帮助快速核对
                cols = list(df.columns)
                display_cols = cols[:5] + ["..."] if len(cols) > 5 else cols
                print(f"     -> 字段: {', '.join(display_cols)}")
                
            # --- 类别 B: 读取栅格数据 (TIFF) ---
            elif file_ext in ['.tif', '.tiff'] and HAS_RASTERIO:
                with rasterio.open(file) as src:
                    raster_metadata[file_key] = {
                        "width": src.width, 
                        "height": src.height, 
                        "crs": src.crs.to_string() if src.crs else "Unknown",
                        "bounds": src.bounds
                    }
                    print(f"  🗺️ [栅格] {file_name}")
                    print(f"     -> 像素: {src.width}x{src.height} | CRS: {raster_metadata[file_key]['crs']}")
                    
            # --- 类别 C: 读取矢量数据 (Shapefile) ---
            elif file_ext == '.shp' and HAS_GEOPANDAS:
                gdf = gpd.read_file(file)
                vector_metadata[file_key] = gdf
                print(f"  📍 [矢量] {file_name}")
                print(f"     -> 维度: {gdf.shape} | CRS: {gdf.crs}")
                
        except Exception as e:
            print(f"  ❌ 读取 {file_name} 时发生错误: {e}")
            
    print("-" * 60)
            
print("\n================ 数据扫描与读取结束 ================")
print(f"✅ 成功加载至内存: {len(tabular_data)} 个表格, {len(raster_metadata)} 个栅格元数据, {len(vector_metadata)} 个矢量。")
print("提示：可以通过 tabular_data['文件夹名_文件名.csv'] 来直接调用具体的 DataFrame。")

In [ ]:
import os
import glob
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import shutil

# --- 1. 定义路径 ---
base_dir = r"E:\chuli"
ny_folders = ['NY_independent_variable', 'NY_dependent_variable']
out_dir = os.path.join(base_dir, 'NY_aligned')
os.makedirs(out_dir, exist_ok=True)

# --- 2. 设定标准基底 (以纽约的街景开阔度为准: EPSG:4326, 4571x4419) ---
ref_raster_path = os.path.join(base_dir, 'NY_independent_variable', 'ratio_NY_plus_Openness_clipped.tif')

print("================ [纽约] 开始栅格对齐 ================")
with rasterio.open(ref_raster_path) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_meta = ref.meta.copy()
    print(f"🎯 参照基准: {ref_width}x{ref_height}, CRS: {ref_crs}")

# --- 3. 遍历并对齐所有 NY 栅格 ---
for folder in ny_folders:
    folder_path = os.path.join(base_dir, folder)
    tif_files = glob.glob(os.path.join(folder_path, "*.tif"))
    
    for src_path in tif_files:
        file_name = os.path.basename(src_path)
        dst_path = os.path.join(out_dir, file_name)
        
        with rasterio.open(src_path) as src:
            # 复制参照基底的元数据，但保留原数据的数据类型和无值设定
            kwargs = ref_meta.copy()
            kwargs.update({
                'dtype': src.dtypes[0],
                'nodata': src.nodata if src.nodata is not None else -9999,
                'count': src.count
            })
            
            # 判断是否已经完美对齐
            if src.crs == ref_crs and src.width == ref_width and src.height == ref_height and src.transform == ref_transform:
                print(f"✅ [已对齐, 直接复制] {file_name}")
                shutil.copy(src_path, dst_path)
                continue
                
            print(f"🔄 [重投影&重采样] {file_name} (原: {src.width}x{src.height}, {src.crs})")
            
            # 创建输出文件并执行重投影
            with rasterio.open(dst_path, 'w', **kwargs) as dst:
                for i in range(1, src.count + 1):
                    destination = np.empty((ref_height, ref_width), dtype=src.dtypes[i-1])
                    reproject(
                        source=rasterio.band(src, i),
                        destination=destination,
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=ref_transform,
                        dst_crs=ref_crs,
                        resampling=Resampling.nearest  # 最邻近插值，保护零膨胀事件频次和极值
                    )
                    dst.write(destination, i)

print(f"\n🎉 纽约数据处理完成！所有对齐后的文件已存入: {out_dir}")

In [ ]:
import os
import glob
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import shutil

# --- 1. 定义路径 ---
base_dir = r"E:\chuli"
hk_folders = ['HK_independent_variable', 'HK_dependent_variable']
out_dir = os.path.join(base_dir, 'HK_aligned')
os.makedirs(out_dir, exist_ok=True)

# --- 2. 设定标准基底 (以香港的街景开阔度为准: EPSG:4326, 4844x3916) ---
ref_raster_path = os.path.join(base_dir, 'HK_independent_variable', 'ratio_HK_plus_Openness_clipped.tif')

print("================ [香港] 开始栅格对齐 ================")
with rasterio.open(ref_raster_path) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_meta = ref.meta.copy()
    print(f"🎯 参照基准: {ref_width}x{ref_height}, CRS: {ref_crs}")

# --- 3. 遍历并对齐所有 HK 栅格 ---
for folder in hk_folders:
    folder_path = os.path.join(base_dir, folder)
    tif_files = glob.glob(os.path.join(folder_path, "*.tif"))
    
    for src_path in tif_files:
        file_name = os.path.basename(src_path)
        dst_path = os.path.join(out_dir, file_name)
        
        with rasterio.open(src_path) as src:
            kwargs = ref_meta.copy()
            kwargs.update({
                'dtype': src.dtypes[0],
                'nodata': src.nodata if src.nodata is not None else -9999,
                'count': src.count
            })
            
            if src.crs == ref_crs and src.width == ref_width and src.height == ref_height and src.transform == ref_transform:
                print(f"✅ [已对齐, 直接复制] {file_name}")
                shutil.copy(src_path, dst_path)
                continue
                
            print(f"🔄 [重投影&重采样] {file_name} (原: {src.width}x{src.height}, {src.crs})")
            
            with rasterio.open(dst_path, 'w', **kwargs) as dst:
                for i in range(1, src.count + 1):
                    destination = np.empty((ref_height, ref_width), dtype=src.dtypes[i-1])
                    reproject(
                        source=rasterio.band(src, i),
                        destination=destination,
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=ref_transform,
                        dst_crs=ref_crs,
                        resampling=Resampling.nearest
                    )
                    dst.write(destination, i)

print(f"\n🎉 香港数据处理完成！所有对齐后的文件已存入: {out_dir}")

In [ ]:
import os
import glob
import rasterio

# --- 1. 定义对齐后的文件夹路径 ---
base_dir = r"E:\chuli"
aligned_folders = {
    'NY_aligned': '纽约',
    'HK_aligned': '香港'
}

print("================ 开始执行空间对齐严格质检 ================\n")

for folder, city_name in aligned_folders.items():
    folder_path = os.path.join(base_dir, folder)
    
    if not os.path.exists(folder_path):
        print(f"⚠️ 警告：未找到文件夹 {folder_path}，请确认上一步对齐代码已成功运行。")
        continue
        
    tif_files = glob.glob(os.path.join(folder_path, "*.tif"))
    if not tif_files:
        print(f"⚠️ 警告：{city_name} 文件夹中没有找到 .tif 文件。")
        continue
        
    print(f"🔍 正在核对 【{city_name}】 ({folder}) 中的 {len(tif_files)} 个文件...")
    
    # --- 2. 以第一个文件作为参照基准 ---
    ref_file = tif_files[0]
    with rasterio.open(ref_file) as ref:
        ref_width = ref.width
        ref_height = ref.height
        ref_crs = ref.crs
        ref_transform = ref.transform
        
    print(f"   -> 🎯 确立基准指标: 维度={ref_width}x{ref_height}, CRS={ref_crs}")
    
    # --- 3. 逐一比对所有文件 ---
    mismatch_count = 0
    for file in tif_files:
        file_name = os.path.basename(file)
        try:
            with rasterio.open(file) as src:
                # 检查四大核心空间属性
                is_match = (
                    src.width == ref_width and
                    src.height == ref_height and
                    src.crs == ref_crs and
                    src.transform == ref_transform
                )
                
                if not is_match:
                    mismatch_count += 1
                    print(f"   ❌ [不匹配] {file_name}")
                    print(f"      - 预期: {ref_width}x{ref_height}, {ref_crs}")
                    print(f"      - 实际: {src.width}x{src.height}, {src.crs}")
                    
        except Exception as e:
            print(f"   ❌ 读取 {file_name} 失败: {e}")
            mismatch_count += 1
            
    # --- 4. 输出核对结论 ---
    if mismatch_count == 0:
        print(f"   ✅ 完美！【{city_name}】的所有栅格数据已达成 100% 空间对齐。")
    else:
        print(f"   ⚠️ 注意！【{city_name}】存在 {mismatch_count} 个未对齐文件，请排查。")
        
    print("-" * 60)

print("\n================ 空间对齐质检结束 ================")

In [ ]:
import os
import glob
import rasterio
import numpy as np
import pandas as pd

# --- 1. 定义路径 ---
base_dir = r"E:\chuli"
aligned_folders = {
    'NY_aligned': '纽约',
    'HK_aligned': '香港'
}

print("================ 开始执行栅格降维与矩阵化提取 ================\n")

for folder, city_name in aligned_folders.items():
    folder_path = os.path.join(base_dir, folder)
    tif_files = glob.glob(os.path.join(folder_path, "*.tif"))
    
    if not tif_files:
        continue
        
    print(f"🚀 正在处理 【{city_name}】 数据，共 {len(tif_files)} 个特征层...")
    
    # --- 2. 寻找掩膜基准 (找一个一定能覆盖全城且边界清晰的变量，如开阔度或道路) ---
    # 我们优先寻找带有 'Openness' 的栅格作为有效区域掩膜 (Mask)
    mask_file = next((f for f in tif_files if 'Openness' in f), tif_files[0])
    
    with rasterio.open(mask_file) as src:
        mask_array = src.read(1)
        nodata_val = src.nodata if src.nodata is not None else -9999
        
        # 定义有效像素：非 NaN 且 非 NoData
        if np.isnan(nodata_val):
            valid_mask = ~np.isnan(mask_array)
        else:
            valid_mask = (mask_array != nodata_val) & ~np.isnan(mask_array)
            
        valid_pixel_count = np.sum(valid_mask)
        print(f"   -> 🎯 提取到有效微观空间单元 (Samples) 数量: {valid_pixel_count}")
        
    # --- 3. 逐一提取所有栅格的有效像素 ---
    city_data_dict = {}
    
    for file in tif_files:
        col_name = os.path.splitext(os.path.basename(file))[0] # 去掉.tif后缀作为列名
        
        with rasterio.open(file) as src:
            array = src.read(1)
            # 提取有效区域的像素值，并展平为一维数组
            valid_values = array[valid_mask]
            
            # 处理可能存在的其他 NoData (统一替换为 np.nan 方便后续 Pandas 处理)
            current_nodata = src.nodata
            if current_nodata is not None and not np.isnan(current_nodata):
                valid_values = np.where(valid_values == current_nodata, np.nan, valid_values)
                
            city_data_dict[col_name] = valid_values
            
    # --- 4. 组装成 Pandas DataFrame 并导出 ---
    print(f"   -> 🧩 正在拼装 {city_name} 的 DataFrame...")
    df_city = pd.DataFrame(city_data_dict)
    
    # 填充因变量中的 NaN 为 0 (因为没有犯罪/事故发生的地方，提取出来可能是 NaN，但在泊松/零膨胀逻辑中它是 0)
    # 假设含有 'Freq' 的列是因变量
    y_cols = [col for col in df_city.columns if 'Freq' in col or 'Arrests' in col or 'Accidents' in col]
    for y_col in y_cols:
        df_city[y_col] = df_city[y_col].fillna(0)
        
    # 导出为最终的 CSV 文件
    output_csv = os.path.join(base_dir, f"{city_name}_final_dataset.csv")
    df_city.to_csv(output_csv, index=False)
    
    print(f"   ✅ 【{city_name}】 矩阵化完成！")
    print(f"      - 最终表格维度: {df_city.shape}")
    print(f"      - 文件已保存至: {output_csv}\n")
    
print("================ 矩阵化提取全部结束 ================")

In [ ]:
import pandas as pd
import os

# --- 1. 定义路径与文件名 ---
base_dir = r"E:\chuli"

# 输入文件 (包含中文的旧名字)
ny_input = os.path.join(base_dir, '纽约_final_dataset.csv')
hk_input = os.path.join(base_dir, '香港_final_dataset.csv')

# 输出文件 (纯英文的新名字)
ny_output = os.path.join(base_dir, 'NY_cleaned_dataset.csv')
hk_output = os.path.join(base_dir, 'HK_cleaned_dataset.csv')

print("================ 开始执行空间矩阵数据清洗 ================\n")

# --- 2. 清洗纽约 (NY) 数据 ---
if os.path.exists(ny_input):
    print("🚀 正在读取并清洗纽约数据...")
    df_ny = pd.read_csv(ny_input)
    
    # 填补所有 NaN 为 0 (符合空间缺失即为'无'的物理逻辑)
    df_ny.fillna(0, inplace=True)
    
    # 导出纯英文命名文件
    df_ny.to_csv(ny_output, index=False)
    print(f"   ✅ 纽约数据清洗完成！")
    print(f"      - 维度: {df_ny.shape}")
    print(f"      - 无缺失值: {df_ny.isnull().sum().sum() == 0}")
    print(f"      - 已保存至: {ny_output}\n")
else:
    print(f"⚠️ 未找到纽约输入文件: {ny_input}")

# --- 3. 清洗香港 (HK) 数据 ---
if os.path.exists(hk_input):
    print("🚀 正在读取并清洗香港数据...")
    df_hk = pd.read_csv(hk_input)
    
    # 填补所有 NaN 为 0
    df_hk.fillna(0, inplace=True)
    
    # 导出纯英文命名文件
    df_hk.to_csv(hk_output, index=False)
    print(f"   ✅ 香港数据清洗完成！")
    print(f"      - 维度: {df_hk.shape}")
    print(f"      - 无缺失值: {df_hk.isnull().sum().sum() == 0}")
    print(f"      - 已保存至: {hk_output}\n")
else:
    print(f"⚠️ 未找到香港输入文件: {hk_input}")

print("================ 数据清洗全部结束 ================")

香港数据集：Hurdle-LightGBM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, mean_absolute_error
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. 读取清洗后的数据
# ---------------------------------------------------------
print("================ 1. 数据加载与时间窗构建 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset.csv"
df = pd.read_csv(hk_file)

# ---------------------------------------------------------
# 2. 构建严格的时间窗 (Time Window)
# ---------------------------------------------------------
# 目标变量 Y：未来发生的事故 (2018 - 2019)
df['Y_Target_Future'] = df['Year_2018_Freq'] + df['Year_2019_Freq']

# 核心历史控制变量 X_prior：过去发生的事故 (2014 - 2017)
df['X_Prior_Accidents'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                           df['Year_2016_Freq'] + df['Year_2017_Freq'])

# ---------------------------------------------------------
# 3. 筛选自变量特征 X
# ---------------------------------------------------------
# 提取所有相关的街景、POI、路网等环境特征，并加入历史先验
env_features = [col for col in df.columns if 'ratio_' in col or 'accessibility' in col or 'density' in col or 'road' in col or 'person' in col]
features = env_features + ['X_Prior_Accidents']

X = df[features]
y = df['Y_Target_Future']

# ---------------------------------------------------------
# 4. 严谨的训练集与测试集划分 (80% Train, 20% Test)
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ 样本集划分完成: 训练集 {X_train.shape[0]} 个栅格, 测试集 {X_test.shape[0]} 个栅格")

# =========================================================
# 5. Hurdle 模型第一阶段：零障碍分类器 (发生 vs 不发生)
# =========================================================
print("\n================ 2. 训练第一阶段: 零障碍分类器 ================")
# 将 Y 转换为二分类：0 为未发生，1 为发生 (>0)
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

# 初始化并训练 LightGBM 分类器
clf = lgb.LGBMClassifier(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5)
clf.fit(X_train, y_train_cls)

# 在测试集上评估
y_pred_cls = clf.predict(X_test)
y_prob_cls = clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test_cls, y_prob_cls)
acc_score = accuracy_score(y_test_cls, y_pred_cls)
print(f"📊 分类阶段 (Test Set) -> AUC: {auc_score:.4f} | Accuracy: {acc_score:.4f}")

# =========================================================
# 6. Hurdle 模型第二阶段：截尾回归器 (仅针对发生事故的区域)
# =========================================================
print("\n================ 3. 训练第二阶段: 截尾回归器 ================")
# 仅筛选出训练集中真实发生过事故的样本进行回归训练
X_train_reg = X_train[y_train > 0]
y_train_reg = y_train[y_train > 0]

# 初始化并训练 LightGBM 回归器
reg = lgb.LGBMRegressor(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5)
reg.fit(X_train_reg, y_train_reg)

# 仅在测试集中真实发生的样本上进行回归评估 (理论评估)
X_test_reg = X_test[y_test > 0]
y_test_reg = y_test[y_test > 0]
y_pred_reg = reg.predict(X_test_reg)

r2 = r2_score(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
mae = mean_absolute_error(y_test_reg, y_pred_reg)
print(f"📊 回归阶段 (Test Set, y>0) -> R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

# =========================================================
# 7. 两阶段联合输出最终预测期望 (Hurdle Expectation)
# =========================================================
print("\n================ 4. Hurdle 联合预测性能评估 ================")
# 联合预测公式: E[Y] = P(Y>0) * E[Y|Y>0]
y_pred_final = clf.predict_proba(X_test)[:, 1] * np.maximum(0, reg.predict(X_test)) # 确保回归预测下限为0

# 评估最终联合预测效果
final_r2 = r2_score(y_test, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_mae = mean_absolute_error(y_test, y_pred_final)

print(f"🏆 Hurdle 最终预测 (Test Set) -> R2: {final_r2:.4f} | RMSE: {final_rmse:.4f} | MAE: {final_mae:.4f}")
print("================ 运行结束 ================")

In [ ]:
import os
import glob
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- 1. 定义路径 ---
base_dir = r"E:\chuli"
hk_aligned_dir = os.path.join(base_dir, "HK_aligned")
out_csv = os.path.join(base_dir, "HK_cleaned_dataset_coarse.csv")

tif_files = glob.glob(os.path.join(hk_aligned_dir, "*.tif"))

# --- 2. 设定聚合倍数 (Scale Factor) ---
scale_factor = 5  # 将 5x5 的像素块聚合成 1 个大像素

print(f"================ 开始执行空间降采样 (Warp聚合，尺度扩大 {scale_factor} 倍) ================\n")

# --- 3. 确立新的空间维度与变换矩阵 (Transform) ---
mask_file = next((f for f in tif_files if 'Openness' in f), tif_files[0])
with rasterio.open(mask_file) as src:
    new_height = int(src.height / scale_factor)
    new_width = int(src.width / scale_factor)
    
    # 计算大栅格的新仿射变换矩阵
    new_transform = src.transform * Affine.scale(src.width / new_width, src.height / new_height)
    
    print(f"🎯 原始维度: {src.height} x {src.width}")
    print(f"🎯 聚合后维度: {new_height} x {new_width}")
    
    # 使用 reproject 提取聚合后的掩膜 (基于平均值)
    mask_array = np.empty((new_height, new_width), dtype=np.float32)
    reproject(
        source=rasterio.band(src, 1),
        destination=mask_array,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=new_transform,
        dst_crs=src.crs,
        resampling=Resampling.average
    )
    
    # 定义有效区域：非 NaN 且 不等于原 NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    valid_mask = ~np.isnan(mask_array) & (mask_array != nodata_val)
    print(f"   -> 新尺度下的有效样本 (Samples) 数量: {np.sum(valid_mask)}")

# --- 4. 逐一聚合读取所有特征 ---
city_data_dict = {}

for file in tif_files:
    col_name = os.path.splitext(os.path.basename(file))[0]
    
    with rasterio.open(file) as src:
        # 事故频次用“求和 (sum)”，建成环境特征用“求均值 (average)”
        if 'Freq' in col_name or 'Accidents' in col_name or 'Arrests' in col_name:
            resample_method = Resampling.sum
        else:
            resample_method = Resampling.average
            
        # 准备空的目标数组
        dest_array = np.empty((new_height, new_width), dtype=np.float32)
        
        # 执行重投影与重采样
        reproject(
            source=rasterio.band(src, 1),
            destination=dest_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=new_transform,
            dst_crs=src.crs,
            resampling=resample_method
        )
        
        # 提取有效区域像素并展平
        valid_values = dest_array[valid_mask]
        
        # 统一将由于降采样或原始缺失产生的 NaN 替换为 0
        valid_values = np.nan_to_num(valid_values, nan=0.0)
        
        city_data_dict[col_name] = valid_values

# --- 5. 导出大尺度 CSV ---
print("\n🧩 正在拼装聚合后的 DataFrame...")
df_coarse = pd.DataFrame(city_data_dict)
df_coarse.to_csv(out_csv, index=False)

print(f"✅ 大尺度聚合数据已生成！")
print(f"   - 表格维度: {df_coarse.shape}")
print(f"   - 已保存至: {out_csv}")
print("================ 聚合流程结束 ================")

In [ ]:
import os
import glob
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- 1. 定义路径 ---
base_dir = r"E:\chuli"
hk_aligned_dir = os.path.join(base_dir, "HK_aligned")
out_csv = os.path.join(base_dir, "HK_cleaned_dataset_coarse.csv")

tif_files = glob.glob(os.path.join(hk_aligned_dir, "*.tif"))

# --- 2. 设定聚合倍数 (Scale Factor) ---
scale_factor = 5  # 将 5x5 的像素块聚合成 1 个大像素

print(f"================ 开始执行空间降采样 (Warp聚合，尺度扩大 {scale_factor} 倍) ================\n")

# --- 3. 确立新的空间维度与变换矩阵 (Transform) ---
mask_file = next((f for f in tif_files if 'Openness' in f), tif_files[0])
with rasterio.open(mask_file) as src:
    new_height = int(src.height / scale_factor)
    new_width = int(src.width / scale_factor)
    
    # 计算大栅格的新仿射变换矩阵
    new_transform = src.transform * Affine.scale(src.width / new_width, src.height / new_height)
    
    print(f"🎯 原始维度: {src.height} x {src.width}")
    print(f"🎯 聚合后维度: {new_height} x {new_width}")
    
    # 使用 reproject 提取聚合后的掩膜 (基于平均值)
    mask_array = np.empty((new_height, new_width), dtype=np.float32)
    reproject(
        source=rasterio.band(src, 1),
        destination=mask_array,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=new_transform,
        dst_crs=src.crs,
        resampling=Resampling.average
    )
    
    # 定义有效区域：非 NaN 且 不等于原 NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    valid_mask = ~np.isnan(mask_array) & (mask_array != nodata_val)
    print(f"   -> 新尺度下的有效样本 (Samples) 数量: {np.sum(valid_mask)}")

# --- 4. 逐一聚合读取所有特征 ---
city_data_dict = {}

for file in tif_files:
    col_name = os.path.splitext(os.path.basename(file))[0]
    
    with rasterio.open(file) as src:
        # 事故频次用“求和 (sum)”，建成环境特征用“求均值 (average)”
        if 'Freq' in col_name or 'Accidents' in col_name or 'Arrests' in col_name:
            resample_method = Resampling.sum
        else:
            resample_method = Resampling.average
            
        # 准备空的目标数组
        dest_array = np.empty((new_height, new_width), dtype=np.float32)
        
        # 执行重投影与重采样
        reproject(
            source=rasterio.band(src, 1),
            destination=dest_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=new_transform,
            dst_crs=src.crs,
            resampling=resample_method
        )
        
        # 提取有效区域像素并展平
        valid_values = dest_array[valid_mask]
        
        # 统一将由于降采样或原始缺失产生的 NaN 替换为 0
        valid_values = np.nan_to_num(valid_values, nan=0.0)
        
        city_data_dict[col_name] = valid_values

# --- 5. 导出大尺度 CSV ---
print("\n🧩 正在拼装聚合后的 DataFrame...")
df_coarse = pd.DataFrame(city_data_dict)
df_coarse.to_csv(out_csv, index=False)

print(f"✅ 大尺度聚合数据已生成！")
print(f"   - 表格维度: {df_coarse.shape}")
print(f"   - 已保存至: {out_csv}")
print("================ 聚合流程结束 ================")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, mean_absolute_error
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. 读取新聚合的粗尺度数据
# ---------------------------------------------------------
print("================ 1. 数据加载与时间窗构建 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# ---------------------------------------------------------
# 2. 构建严格的时间窗 (Time Window)
# ---------------------------------------------------------
# 目标变量 Y：未来发生的事故 (2018 - 2019)
df['Y_Target_Future'] = df['Year_2018_Freq'] + df['Year_2019_Freq']

# 核心历史控制变量 X_prior：过去发生的事故 (2014 - 2017)
df['X_Prior_Accidents'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                           df['Year_2016_Freq'] + df['Year_2017_Freq'])

# ---------------------------------------------------------
# 3. 筛选自变量特征 X
# ---------------------------------------------------------
env_features = [col for col in df.columns if 'ratio_' in col or 'accessibility' in col or 'density' in col or 'road' in col or 'person' in col]
features = env_features + ['X_Prior_Accidents']

X = df[features]
y = df['Y_Target_Future']

# ---------------------------------------------------------
# 4. 严谨的训练集与测试集划分 (80% Train, 20% Test)
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ 样本集划分完成: 训练集 {X_train.shape[0]} 个栅格, 测试集 {X_test.shape[0]} 个栅格")

# =========================================================
# 5. Hurdle 模型第一阶段：零障碍分类器
# =========================================================
print("\n================ 2. 训练第一阶段: 零障碍分类器 ================")
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

clf = lgb.LGBMClassifier(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5)
clf.fit(X_train, y_train_cls)

y_pred_cls = clf.predict(X_test)
y_prob_cls = clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test_cls, y_prob_cls)
acc_score = accuracy_score(y_test_cls, y_pred_cls)
print(f"📊 分类阶段 (Test Set) -> AUC: {auc_score:.4f} | Accuracy: {acc_score:.4f}")

# =========================================================
# 6. Hurdle 模型第二阶段：截尾回归器
# =========================================================
print("\n================ 3. 训练第二阶段: 截尾回归器 ================")
X_train_reg = X_train[y_train > 0]
y_train_reg = y_train[y_train > 0]

# 改用 poisson 回归损失函数，更适合计数值预测
reg = lgb.LGBMRegressor(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5, objective='poisson')
reg.fit(X_train_reg, y_train_reg)

X_test_reg = X_test[y_test > 0]
y_test_reg = y_test[y_test > 0]
y_pred_reg = reg.predict(X_test_reg)

r2 = r2_score(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
mae = mean_absolute_error(y_test_reg, y_pred_reg)
print(f"📊 回归阶段 (Test Set, y>0) -> R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

# =========================================================
# 7. 两阶段联合预测
# =========================================================
print("\n================ 4. Hurdle 联合预测性能评估 ================")
y_pred_final = clf.predict_proba(X_test)[:, 1] * np.maximum(0, reg.predict(X_test))

final_r2 = r2_score(y_test, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_mae = mean_absolute_error(y_test, y_pred_final)

print(f"🏆 Hurdle 最终预测 (Test Set) -> R2: {final_r2:.4f} | RMSE: {final_rmse:.4f} | MAE: {final_mae:.4f}")
print("================ 运行结束 ================")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("================ 1. 数据加载与时间窗构建 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# --- 严格时间窗 ---
df['Y_Target_Future'] = df['Year_2018_Freq'] + df['Year_2019_Freq']
df['X_Prior_Accidents'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                           df['Year_2016_Freq'] + df['Year_2017_Freq'])

# --- 筛选特征 ---
env_features = [col for col in df.columns if 'ratio_' in col or 'accessibility' in col or 'density' in col or 'road' in col or 'person' in col]
features = env_features + ['X_Prior_Accidents']

X = df[features]
y = df['Y_Target_Future']

# --- 严谨划分 (80% Train, 20% Test) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ 样本集: 训练集 {X_train.shape[0]}, 测试集 {X_test.shape[0]}")

# =========================================================
# 5. Hurdle 第一阶段：XGBoost 零障碍分类器
# =========================================================
print("\n================ 2. 训练第一阶段: XGBoost 分类器 ================")
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

# 加入正则化防过拟合，树加深
xgb_clf = xgb.XGBClassifier(
    random_state=42, 
    n_estimators=150,      # 增加树的数量
    learning_rate=0.05, 
    max_depth=6,           # 加深树的深度
    subsample=0.8,         # 随机采样样本比例
    colsample_bytree=0.8,  # 随机采样特征比例
    reg_alpha=0.1,         # L1 正则化
    reg_lambda=1.0,        # L2 正则化
    eval_metric='logloss'
)
xgb_clf.fit(X_train, y_train_cls)

y_pred_cls = xgb_clf.predict(X_test)
y_prob_cls = xgb_clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test_cls, y_prob_cls)
print(f"📊 分类阶段 (Test Set) -> AUC: {auc_score:.4f} | Accuracy: {accuracy_score(y_test_cls, y_pred_cls):.4f}")

# =========================================================
# 6. Hurdle 第二阶段：XGBoost 截尾回归器 (泊松分布)
# =========================================================
print("\n================ 3. 训练第二阶段: XGBoost 截尾回归器 ================")
X_train_reg = X_train[y_train > 0]
y_train_reg = y_train[y_train > 0]

xgb_reg = xgb.XGBRegressor(
    objective='count:poisson',  # 严格指定泊松回归损失函数
    random_state=42, 
    n_estimators=150, 
    learning_rate=0.03,         # 降低学习率，提升精度
    max_depth=6, 
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0
)
xgb_reg.fit(X_train_reg, y_train_reg)

X_test_reg = X_test[y_test > 0]
y_test_reg = y_test[y_test > 0]
y_pred_reg = xgb_reg.predict(X_test_reg)

r2 = r2_score(y_test_reg, y_pred_reg)
print(f"📊 回归阶段 (Test Set, y>0) -> R2: {r2:.4f} | RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.4f}")

# =========================================================
# 7. 两阶段联合预测
# =========================================================
print("\n================ 4. Hurdle 联合预测性能评估 ================")
y_pred_final = xgb_clf.predict_proba(X_test)[:, 1] * np.maximum(0, xgb_reg.predict(X_test))

final_r2 = r2_score(y_test, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_mae = mean_absolute_error(y_test, y_pred_final)

print(f"🏆 Hurdle-XGBoost 最终预测 (Test Set) -> R2: {final_r2:.4f} | RMSE: {final_rmse:.4f} | MAE: {final_mae:.4f}")
print("================ 运行结束 ================")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

print("================ 1. 数据加载与时间窗构建 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# --- 严格时间窗 ---
df['Y_Target_Future'] = df['Year_2018_Freq'] + df['Year_2019_Freq']
df['X_Prior_Accidents'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                           df['Year_2016_Freq'] + df['Year_2017_Freq'])

# --- 筛选特征 ---
env_features = [col for col in df.columns if 'ratio_' in col or 'accessibility' in col or 'density' in col or 'road' in col or 'person' in col]
features = env_features + ['X_Prior_Accidents']

X = df[features]
y = df['Y_Target_Future']

# --- 严谨划分 (80% Train, 20% Test) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ 样本集: 训练集 {X_train.shape[0]} 个栅格, 测试集 {X_test.shape[0]} 个栅格")

# =========================================================
# 5. Hurdle 第一阶段：随机森林 (Random Forest) 分类器
# =========================================================
print("\n================ 2. 训练第一阶段: 随机森林 分类器 ================")
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

# 限制 max_depth 防止过拟合，并开启多线程加速 n_jobs=-1
rf_clf = RandomForestClassifier(random_state=42, n_estimators=150, max_depth=6, n_jobs=-1)
rf_clf.fit(X_train, y_train_cls)

y_pred_cls = rf_clf.predict(X_test)
y_prob_cls = rf_clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test_cls, y_prob_cls)
print(f"📊 分类阶段 (Test Set) -> AUC: {auc_score:.4f} | Accuracy: {accuracy_score(y_test_cls, y_pred_cls):.4f}")

# =========================================================
# 6. Hurdle 第二阶段：随机森林 (Random Forest) 截尾回归器
# =========================================================
print("\n================ 3. 训练第二阶段: 随机森林 截尾回归器 ================")
X_train_reg = X_train[y_train > 0]
y_train_reg = y_train[y_train > 0]

# 尝试使用 poisson 准则 (在较新的 scikit-learn 版本中支持) 
# 如果报错，可以将其删去或改为默认的 'squared_error'
try:
    rf_reg = RandomForestRegressor(random_state=42, n_estimators=150, max_depth=6, criterion='poisson', n_jobs=-1)
    rf_reg.fit(X_train_reg, y_train_reg)
except Exception as e:
    print("  -> 提示: 当前 sklearn 不支持 poisson criterion，自动回退到默认 MSE 损失。")
    rf_reg = RandomForestRegressor(random_state=42, n_estimators=150, max_depth=6, n_jobs=-1)
    rf_reg.fit(X_train_reg, y_train_reg)

X_test_reg = X_test[y_test > 0]
y_test_reg = y_test[y_test > 0]
y_pred_reg = rf_reg.predict(X_test_reg)

r2 = r2_score(y_test_reg, y_pred_reg)
print(f"📊 回归阶段 (Test Set, y>0) -> R2: {r2:.4f} | RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.4f}")

# =========================================================
# 7. 两阶段联合预测
# =========================================================
print("\n================ 4. Hurdle 联合预测性能评估 ================")
y_pred_final = rf_clf.predict_proba(X_test)[:, 1] * np.maximum(0, rf_reg.predict(X_test))

final_r2 = r2_score(y_test, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_mae = mean_absolute_error(y_test, y_pred_final)

print(f"🏆 Hurdle-RandomForest 最终预测 (Test Set) -> R2: {final_r2:.4f} | RMSE: {final_rmse:.4f} | MAE: {final_mae:.4f}")
print("================ 运行结束 ================")

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("================ 1. 消融实验：加载与清洗 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# --- 目标变量 ---
df['Y_Target_Future'] = df['Year_2018_Freq'] + df['Year_2019_Freq']

# --- 🎯 消融核心：剥离历史先验 ---
# 我们只提取纯粹的微观环境特征（视觉特征、POI可达性、道路密度等）
features_ablation = [col for col in df.columns if 'ratio_' in col or 'accessibility' in col or 'density' in col or 'road' in col or 'person' in col]

X = df[features_ablation]
y = df['Y_Target_Future']

# 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ 剥离历史先验完成！当前纯环境特征数量: {len(features_ablation)}")

# =========================================================
# 2. 初始化分类器与回归器 (与之前保持完全一致的超参数)
# =========================================================
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

X_train_reg = X_train[y_train > 0]
y_train_reg = y_train[y_train > 0]
X_test_reg = X_test[y_test > 0]
y_test_reg = y_test[y_test > 0]

models = {
    "LightGBM": {
        "clf": lgb.LGBMClassifier(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5, n_jobs=-1),
        "reg": lgb.LGBMRegressor(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5, objective='poisson', n_jobs=-1)
    },
    "XGBoost": {
        "clf": xgb.XGBClassifier(random_state=42, n_estimators=150, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, eval_metric='logloss', n_jobs=-1),
        "reg": xgb.XGBRegressor(objective='count:poisson', random_state=42, n_estimators=150, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1)
    },
    "Random Forest": {
        "clf": RandomForestClassifier(random_state=42, n_estimators=150, max_depth=6, n_jobs=-1),
        "reg": RandomForestRegressor(random_state=42, n_estimators=150, max_depth=6, n_jobs=-1)
    }
}

# =========================================================
# 3. 循环执行三大模型的 Hurdle 架构并输出对比
# =========================================================
print("\n================ 2. 启动三大引擎消融测试 ================")
for model_name, engine in models.items():
    print(f"\n--- 🚀 正在测试: {model_name} (无历史先验) ---")
    clf = engine["clf"]
    reg = engine["reg"]
    
    # 阶段一：零障碍分类
    clf.fit(X_train, y_train_cls)
    y_prob_cls = clf.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test_cls, y_prob_cls)
    
    # 阶段二：截尾回归
    reg.fit(X_train_reg, y_train_reg)
    y_pred_reg = reg.predict(X_test_reg)
    r2_reg = r2_score(y_test_reg, y_pred_reg)
    
    # 联合预测
    y_pred_final = clf.predict_proba(X_test)[:, 1] * np.maximum(0, reg.predict(X_test))
    final_r2 = r2_score(y_test, y_pred_final)
    final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
    
    print(f"   📊 [阶段一] 分类 AUC : {auc_score:.4f}")
    print(f"   📊 [阶段二] 回归 R2  : {r2_reg:.4f}")
    print(f"   🏆 [总成绩] 联合 R2  : {final_r2:.4f} | RMSE: {final_rmse:.4f}")

print("\n================ 实验结束 ================")

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("================ 1. 消融实验：加载与清洗 (仅保留历史先验) ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# --- 目标变量 ---
df['Y_Target_Future'] = df['Year_2018_Freq'] + df['Year_2019_Freq']

# --- 自变量 ---
df['X_Prior_Accidents'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                           df['Year_2016_Freq'] + df['Year_2017_Freq'])

# 🎯 消融核心：剥离所有街景和建成环境特征，仅留 1 个历史特征！
features_ablation_history_only = ['X_Prior_Accidents']

X = df[features_ablation_history_only]
y = df['Y_Target_Future']

# 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ 剥离环境特征完成！当前仅使用 1 个特征: {features_ablation_history_only}")

# =========================================================
# 2. 初始化分类器与回归器 (与全量模型参数一致)
# =========================================================
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

X_train_reg = X_train[y_train > 0]
y_train_reg = y_train[y_train > 0]
X_test_reg = X_test[y_test > 0]
y_test_reg = y_test[y_test > 0]

models = {
    "LightGBM": {
        "clf": lgb.LGBMClassifier(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5, n_jobs=-1),
        "reg": lgb.LGBMRegressor(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=5, objective='poisson', n_jobs=-1)
    },
    "XGBoost": {
        "clf": xgb.XGBClassifier(random_state=42, n_estimators=150, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, eval_metric='logloss', n_jobs=-1),
        "reg": xgb.XGBRegressor(objective='count:poisson', random_state=42, n_estimators=150, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1)
    },
    "Random Forest": {
        "clf": RandomForestClassifier(random_state=42, n_estimators=150, max_depth=6, n_jobs=-1),
        "reg": RandomForestRegressor(random_state=42, n_estimators=150, max_depth=6, n_jobs=-1)
    }
}

# =========================================================
# 3. 循环执行三大模型的 Hurdle 架构并输出对比
# =========================================================
print("\n================ 2. 启动三大引擎消融测试 (历史基线) ================")
for model_name, engine in models.items():
    print(f"\n--- 🚀 正在测试: {model_name} (仅依赖历史先验) ---")
    clf = engine["clf"]
    reg = engine["reg"]
    
    # 阶段一：零障碍分类
    clf.fit(X_train, y_train_cls)
    y_prob_cls = clf.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test_cls, y_prob_cls)
    
    # 阶段二：截尾回归
    reg.fit(X_train_reg, y_train_reg)
    y_pred_reg = reg.predict(X_test_reg)
    r2_reg = r2_score(y_test_reg, y_pred_reg)
    
    # 联合预测
    y_pred_final = clf.predict_proba(X_test)[:, 1] * np.maximum(0, reg.predict(X_test))
    final_r2 = r2_score(y_test, y_pred_final)
    final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
    
    print(f"   📊 [阶段一] 分类 AUC : {auc_score:.4f}")
    print(f"   📊 [阶段二] 回归 R2  : {r2_reg:.4f}")
    print(f"   🏆 [总成绩] 联合 R2  : {final_r2:.4f} | RMSE: {final_rmse:.4f}")

print("\n================ 实验结束 ================")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, roc_auc_score
import warnings

warnings.filterwarnings('ignore')

# 1. 数据准备 (保持原始颗粒度)
print("================ 1. 原始微观尺度数据准备 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

df['Y_Target'] = df['Year_2018_Freq'] + df['Year_2019_Freq']
df['X_Prior'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                 df['Year_2016_Freq'] + df['Year_2017_Freq'])

# 定义特征组
env_features = [col for col in df.columns if any(k in col for k in ['ratio_', 'accessibility', 'density', 'road'])]
history_feature = ['X_Prior']

X = df[env_features + history_feature]
y = df['Y_Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =========================================================
# 2. 定义 Hurdle 核心对比逻辑
# =========================================================
def evaluate_hurdle_contribution(name, clf, reg):
    print(f"\n--- 🚀 正在评估模型: {name} ---")
    
    # A. 仅历史先验的分类能力 (基准)
    y_tr_cls = (y_train > 0).astype(int)
    y_te_cls = (y_test > 0).astype(int)
    
    clf.fit(X_train[history_feature], y_tr_cls)
    auc_history = roc_auc_score(y_te_cls, clf.predict_proba(X_test[history_feature])[:, 1])
    
    # B. 仅街景环境的分类能力 (核心亮点)
    clf.fit(X_train[env_features], y_tr_cls)
    auc_env = roc_auc_score(y_te_cls, clf.predict_proba(X_test[env_features])[:, 1])
    
    # C. 全量模型的 Hurdle R2
    # Stage 1
    clf.fit(X_train, y_tr_cls)
    prob_test = clf.predict_proba(X_test)[:, 1]
    auc_full = roc_auc_score(y_te_cls, prob_test)
    
    # Stage 2 (回归)
    X_tr_reg = X_train[y_train > 0]
    y_tr_reg = y_train[y_train > 0]
    reg.fit(X_tr_reg, y_tr_reg)
    
    y_pred_final = prob_test * np.maximum(0, reg.predict(X_test))
    final_r2 = r2_score(y_test, y_pred_final)
    
    return {
        'Model': name,
        'History AUC': auc_history,
        'Env(Street) AUC': auc_env,
        'Full Model AUC': auc_full,
        'Final Hurdle R2': final_r2
    }

# =========================================================
# 3. 运行三大引擎对比
# =========================================================
# 超参数保持中庸，重点看特征贡献
models_config = [
    ("XGBoost", 
     xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42),
     xgb.XGBRegressor(objective='count:poisson', n_estimators=300, max_depth=6, learning_rate=0.03, random_state=42)),
    
    ("LightGBM",
     lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05, random_state=42),
     lgb.LGBMRegressor(objective='poisson', n_estimators=300, num_leaves=31, learning_rate=0.03, random_state=42)),
    
    ("Random Forest",
     RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
     RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42))
]

results = []
for name, c, r in models_config:
    results.append(evaluate_hurdle_contribution(name, c, r))

# 4. 结果汇总
res_df = pd.DataFrame(results)
print("\n================ 🏆 策略优越性与街景贡献汇总表 ================")
print(res_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, roc_auc_score, accuracy_score, 
                             precision_score, recall_score, f1_score, 
                             mean_squared_error, mean_absolute_error)
import shap
import warnings

warnings.filterwarnings('ignore')

# =========================================================
# 1. 数据准备
# =========================================================
print("================ 1. 数据加载与对齐 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# 目标变量与历史先验
df['Y_Target'] = df['Year_2018_Freq'] + df['Year_2019_Freq']
df['X_Prior'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                 df['Year_2016_Freq'] + df['Year_2017_Freq'])

# 提取特征集
env_features = [col for col in df.columns if any(k in col for k in ['ratio_', 'accessibility', 'density', 'road'])]
history_feature = ['X_Prior']
full_features = env_features + history_feature

# 划分全量数据集
X_full = df[full_features]
y = df['Y_Target']
X_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)

# =========================================================
# 2. 核心评估函数 (输出丰富指标)
# =========================================================
def evaluate_hurdle(name, experiment_type, clf, reg, features):
    X_tr = X_train_full[features]
    X_te = X_test_full[features]
    
    y_tr_cls = (y_train > 0).astype(int)
    y_te_cls = (y_test > 0).astype(int)
    
    # 阶段一：分类
    clf.fit(X_tr, y_tr_cls)
    prob_test = clf.predict_proba(X_te)[:, 1]
    pred_cls = clf.predict(X_te)
    
    auc = roc_auc_score(y_te_cls, prob_test)
    acc = accuracy_score(y_te_cls, pred_cls)
    prec = precision_score(y_te_cls, pred_cls, zero_division=0)
    rec = recall_score(y_te_cls, pred_cls, zero_division=0)
    f1 = f1_score(y_te_cls, pred_cls, zero_division=0)
    
    # 阶段二：回归
    mask_reg = y_train > 0
    X_tr_reg = X_tr[mask_reg]
    y_tr_reg = y_train[mask_reg]
    
    reg.fit(X_tr_reg, y_tr_reg)
    pred_reg_test = reg.predict(X_te)
    
    # 回归阶段内部的 R2
    mask_test_reg = y_test > 0
    if mask_test_reg.sum() > 0:
        stage2_r2 = r2_score(y_test[mask_test_reg], pred_reg_test[mask_test_reg])
    else:
        stage2_r2 = np.nan
        
    # 最终联合预测
    y_pred_final = prob_test * np.maximum(0, pred_reg_test)
    final_r2 = r2_score(y_test, y_pred_final)
    final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
    final_mae = mean_absolute_error(y_test, y_pred_final)
    
    return {
        'Model': name,
        'Experiment': experiment_type,
        'Cls_AUC': auc,
        'Cls_F1': f1,
        'Cls_Recall': rec,
        'Reg_Stage2_R2': stage2_r2,
        'Final_R2': final_r2,
        'Final_RMSE': final_rmse,
        'Final_MAE': final_mae
    }, clf # 返回训练好的分类器，用于后面跑 SHAP

# =========================================================
# 3. 运行三大消融实验
# =========================================================
print("\n================ 2. 开始执行全面消融对比实验 ================")
results = []
full_classifiers = {} # 保存全量模型的分类器跑 SHAP

# --- A. XGBoost ---
# 1. 纯历史组 (特意压低树深，反映单变量外推的局限性)
xgb_c_hist = xgb.XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
xgb_r_hist = xgb.XGBRegressor(objective='count:poisson', n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
res, _ = evaluate_hurdle("XGBoost", "History Only", xgb_c_hist, xgb_r_hist, history_feature)
results.append(res)

# 2. 纯街景组
xgb_c_env = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)
xgb_r_env = xgb.XGBRegressor(objective='count:poisson', n_estimators=200, max_depth=6, learning_rate=0.03, random_state=42)
res, _ = evaluate_hurdle("XGBoost", "Street View Only", xgb_c_env, xgb_r_env, env_features)
results.append(res)

# 3. 全量模型
xgb_c_full = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)
xgb_r_full = xgb.XGBRegressor(objective='count:poisson', n_estimators=300, max_depth=8, learning_rate=0.03, random_state=42)
res, clf_full = evaluate_hurdle("XGBoost", "Full Model", xgb_c_full, xgb_r_full, full_features)
results.append(res)
full_classifiers["XGBoost"] = clf_full

# --- B. LightGBM ---
lgb_c_hist = lgb.LGBMClassifier(n_estimators=150, max_depth=3, num_leaves=7, learning_rate=0.05, random_state=42)
lgb_r_hist = lgb.LGBMRegressor(objective='poisson', n_estimators=150, max_depth=3, num_leaves=7, learning_rate=0.05, random_state=42)
res, _ = evaluate_hurdle("LightGBM", "History Only", lgb_c_hist, lgb_r_hist, history_feature)
results.append(res)

lgb_c_env = lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05, random_state=42)
lgb_r_env = lgb.LGBMRegressor(objective='poisson', n_estimators=200, num_leaves=31, learning_rate=0.03, random_state=42)
res, _ = evaluate_hurdle("LightGBM", "Street View Only", lgb_c_env, lgb_r_env, env_features)
results.append(res)

lgb_c_full = lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05, random_state=42)
lgb_r_full = lgb.LGBMRegressor(objective='poisson', n_estimators=300, num_leaves=63, learning_rate=0.03, random_state=42)
res, clf_full = evaluate_hurdle("LightGBM", "Full Model", lgb_c_full, lgb_r_full, full_features)
results.append(res)
full_classifiers["LightGBM"] = clf_full

# --- C. Random Forest ---
rf_c_hist = RandomForestClassifier(n_estimators=150, max_depth=3, random_state=42)
rf_r_hist = RandomForestRegressor(n_estimators=150, max_depth=3, random_state=42)
res, _ = evaluate_hurdle("Random Forest", "History Only", rf_c_hist, rf_r_hist, history_feature)
results.append(res)

rf_c_env = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_r_env = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
res, _ = evaluate_hurdle("Random Forest", "Street View Only", rf_c_env, rf_r_env, env_features)
results.append(res)

rf_c_full = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42)
rf_r_full = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
res, clf_full = evaluate_hurdle("Random Forest", "Full Model", rf_c_full, rf_r_full, full_features)
results.append(res)
full_classifiers["Random Forest"] = clf_full

# 打印多维度对比透视表
df_results = pd.DataFrame(results)
print("\n================ 🏆 论文级多维评估指标矩阵 ================")
pivot_df = df_results.pivot(index='Model', columns='Experiment', values=['Cls_AUC', 'Cls_Recall', 'Final_R2', 'Final_MAE'])

columns_order = ['Street View Only', 'History Only', 'Full Model']
for metric in ['Cls_AUC', 'Cls_Recall', 'Final_R2', 'Final_MAE']:
    print(f"\n--- 评估指标: {metric} ---")
    if all(col in pivot_df[metric].columns for col in columns_order):
        print(pivot_df[metric][columns_order].to_string())
    else:
        print(pivot_df[metric].to_string())

# =========================================================
# 4. 全量模型 SHAP (分类阶段) - 挖掘核心致因 (终极防弹版)
# =========================================================
print("\n================ 3. 开启 SHAP 全局解释 (提取分类阶段 Top 10) ================")

for name, clf in full_classifiers.items():
    print(f"\n🚀 正在计算 {name} 的 SHAP 归因...")
    explainer = shap.TreeExplainer(clf)
    
    # 抽取 3000 个背景样本加快计算速度
    X_sample = X_train_full.sample(n=3000, random_state=42)
    shap_vals = explainer.shap_values(X_sample)
    
    # 统一处理异构算法的 SHAP 数组格式
    if isinstance(shap_vals, list):
        shap_vals_target = shap_vals[1] # RandomForest / LGBM 通常返回 List
    else:
        # XGBoost 版本不同，可能是 2D 或 3D 数组
        if len(shap_vals.shape) == 3:
            shap_vals_target = shap_vals[:, :, 1]
        else:
            shap_vals_target = shap_vals
            
    # 计算均值并严谨压扁为一维
    mean_abs_shap = np.abs(shap_vals_target).mean(axis=0)
    if len(mean_abs_shap.shape) > 1:
        mean_abs_shap = np.squeeze(mean_abs_shap)
        
    # 获取排序索引
    top_indices = np.argsort(mean_abs_shap)[::-1][:10]
    
    print(f"🏆 {name} 风险判别阶段 Top 10 核心致灾因子:")
    for rank, idx in enumerate(top_indices):
        # 终极类型转换修复：确保取出的是纯粹的 Python int
        try:
            idx_int = int(np.squeeze(idx).item())
        except:
            idx_int = int(idx)
            
        feature_name = full_features[idx_int]
        importance = mean_abs_shap[idx_int]
        
        # 美化输出，分类打标签
        if feature_name == 'X_Prior':
            feature_name = f"⭐ {feature_name} (历史流量惯性)"
        else:
            feature_name = f"🌿 {feature_name} (微观视觉/建成环境)"
            
        print(f"   Top {rank+1:2d}: {feature_name: <55} | SHAP 平均贡献: {importance:.4f}")

print("\n================ 全部实验与分析生成完毕 ================")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import shap
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# 设置绘图中文字体为微软雅黑，防止图表中的减号和中文乱码
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei'] 
plt.rcParams['axes.unicode_minus'] = False 

# =========================================================
# 1. 数据准备 (保持原始微观尺度)
# =========================================================
print("================ 1. 核心数据与特征准备 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# 目标变量 (二分类：有没有事故，对应 Hurdle 第一阶段)
df['Y_Target_Binary'] = ((df['Year_2018_Freq'] + df['Year_2019_Freq']) > 0).astype(int)
df['X_Prior'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                 df['Year_2016_Freq'] + df['Year_2017_Freq'])

# 准备全量特征集
env_features = [col for col in df.columns if any(k in col for k in ['ratio_', 'accessibility', 'density', 'road'])]
history_feature = ['X_Prior']
full_features = env_features + history_feature

X = df[full_features]
y = df['Y_Target_Binary']

# 保持与消融实验一致的划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 定义要绘制 SHAP 依赖图的核心街景特征（需精确匹配 CSV 列名）
target_shap_features = [
    'ratio_HK_plus_Traffic_flow_clipped',  # 车流视觉占比
    'ratio_HK_plus_Greening_degree_clipped' # 绿视率
]

# =========================================================
# 2. 重新快速训练全量分类器 (确保参数最优)
# =========================================================
print("\n================ 2. 重新训练 Hurdle Stage 1 全量分类器 ================")

# 使用全量模型组的超参数
full_clf_models = {
    "XGBoost": xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
}

for name, model in full_clf_models.items():
    print(f"正在训练 {name}...")
    model.fit(X_train, y_train)

# =========================================================
# 3. 计算 SHAP 并绘制依赖图 (终极防弹版)
# =========================================================
print("\n================ 3. 开启全量模型 SHAP 绘图流程 ================")

# 抽取 2000 个测试样本进行 SHAP 计算，兼顾速度与图表细腻度
X_shap = X_test.sample(n=min(2000, len(X_test)), random_state=42)

for model_name, clf_model in full_clf_models.items():
    print(f"\n🚀 正在计算 {model_name} 的 SHAP 依赖图...")
    
    # 初始化解释器
    explainer = shap.TreeExplainer(clf_model)
    shap_values_raw = explainer.shap_values(X_shap)
    
    # 统一异构算法的 SHAP 输出格式
    if isinstance(shap_values_raw, list):
        shap_values_to_plot = shap_values_raw[1]
    else:
        # 有些版本的 XGBoost 对于二分类返回三维数组 (样本, 特征, 类别)，有些返回二维
        if len(shap_values_raw.shape) == 3:
            shap_values_to_plot = shap_values_raw[:, :, 1]
        else:
            shap_values_to_plot = shap_values_raw
            
    # --- 核心街景特征依赖图绘制 ---
    for feature_to_plot in target_shap_features:
        # 检查特征是否存在于当前模型的特征集中
        if feature_to_plot not in full_features:
            print(f"警告: 特征 {feature_to_plot} 不在模型特征集中，跳过。")
            continue
            
        print(f"   >>> 正在绘制: {feature_to_plot} VS X_Prior (历史事故) 交互依赖图...")
        
        # 创建画布
        plt.figure(figsize=(8, 5))
        
        # 极简防弹版绘图调用，去除了所有可能引发 TypeError 的外围参数
        shap.dependence_plot(
            feature_to_plot, 
            shap_values_to_plot, 
            X_shap, 
            interaction_index='X_Prior', # 设置颜色交互项，观察历史先验的影响
            show=False # 暂不显示，以便我们重写标题和标签
        )
        
        # 美化图表标签，使其学术化
        current_feature_label = feature_to_plot.replace('ratio_HK_plus_', '').replace('_clipped', '').replace('_', ' ').title()
        
        plt.xlabel(f'{current_feature_label} (Feature Value)', fontsize=12, fontweight='bold')
        plt.ylabel('SHAP Value (Impact on Risk)', fontsize=12, fontweight='bold')
        plt.title(f'Nonlinear Threshold of {current_feature_label}\nModel: {model_name} (Risk Identification)', fontsize=14, pad=15)
        
        # 自动调整布局并显示
        plt.tight_layout()
        plt.show() 

print("\n================ 全部图表绘制完成 ================")

In [ ]:
import os
import shap
import numpy as np
import matplotlib
matplotlib.use('Agg') # 开启无头模式，防止连续画图导致 Jupyter 崩溃或卡顿
import matplotlib.pyplot as plt

# =========================================================
# 5. 独立执行块：生成并保存香港三大全量模型的 SHAP 蜂群图
# =========================================================
print("================ 正在生成并保存 SHAP 蜂群图 (Beeswarm) ================")

# 在桌面创建专用的图片保存文件夹
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
hk_pic_dir = os.path.join(desktop_path, "HK_SHAP_Pics")
os.makedirs(hk_pic_dir, exist_ok=True)
print(f"📁 图像保存目录已准备就绪: {hk_pic_dir}")

# 遍历内存中已经训练好的 full_classifiers 字典
for name, clf in full_classifiers.items():
    print(f"📊 正在绘制 {name} 的 SHAP 蜂群图...")
    
    # 初始化树模型解释器
    explainer = shap.TreeExplainer(clf)
    
    # 抽取 3000 个样本以加快画图速度，并防止图面点太密集糊成一团
    # 这里直接调用你上一个 Cell 内存里的 X_train_full
    X_sample = X_train_full.sample(n=3000, random_state=42)
    shap_vals = explainer.shap_values(X_sample)
    
    # 统一处理异构算法的 SHAP 数组格式（提取分类为 1，即发生风险的贡献度）
    if isinstance(shap_vals, list):
        shap_vals_target = shap_vals[1] # RandomForest / LGBM 返回 List
    else:
        # XGBoost 版本差异兼容处理
        if len(shap_vals.shape) == 3:
            shap_vals_target = shap_vals[:, :, 1] 
        else:
            shap_vals_target = shap_vals 
            
    # 初始化画布
    plt.figure(figsize=(10, 8))
    
    # 绘制蜂群图 (max_display 控制图表显示的特征数量，15 个最利于论文排版)
    shap.summary_plot(shap_vals_target, X_sample, max_display=15, show=False)
    
    # 美化标题
    plt.title(f"SHAP Summary: {name} - Full Model (Hong Kong)", fontsize=14, pad=20)
    plt.tight_layout()
    
    # 组合文件名并保存
    safe_name = name.replace(" ", "_")
    img_path = os.path.join(hk_pic_dir, f"SHAP_Beeswarm_HK_{safe_name}.png")
    
    # 保存高清图片并清理内存画布
    plt.savefig(img_path, dpi=300, bbox_inches='tight')
    plt.close('all')

print(f"\n🎉 所有的 SHAP 蜂群图已成功生成！请前往桌面的 【HK_SHAP_Pics】 文件夹中提取！")

In [ ]:
import pandas as pd
import os

# 1. 桌面路径
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
output_path = os.path.join(desktop_path, "双城实验结果汇总.xlsx")

# 2. 构建香港数据
hk_data = {
    '评估阶段': ['第一阶段(分类)']*6 + ['第二阶段(回归)']*6,
    '评价指标': ['Cls_AUC']*3 + ['Cls_Recall']*3 + ['Final_R2']*3 + ['Final_MAE']*3,
    '算法模型': ['LightGBM', 'Random Forest', 'XGBoost'] * 4,
    '仅街景视觉 (Street View Only)': [0.7551, 0.7473, 0.7547, 0.4371, 0.3570, 0.4452, 0.1475, 0.1193, 0.1406, 0.7068, 0.7307, 0.7066],
    '仅历史基线 (History Only)': [0.8052, 0.8058, 0.8046, 0.5779, 0.5783, 0.5779, 0.5148, 0.5204, 0.5274, 0.5524, 0.5534, 0.5499],
    '完整模型 (Full Model)': [0.8359, 0.8347, 0.8362, 0.5728, 0.5612, 0.5788, 0.5023, 0.5159, 0.5041, 0.5422, 0.5506, 0.5397]
}
df_hk = pd.DataFrame(hk_data)

# 3. 构建纽约数据 (XGBoost)
ny_data = {
    '犯罪类型': ['毒品与交通', '毒品与交通', '毒品与交通', '毒品与交通', 
                 '财产犯罪', '财产犯罪', '财产犯罪', '财产犯罪', 
                 '公共秩序', '公共秩序', '公共秩序', '公共秩序', 
                 '性犯罪/未成年', '性犯罪/未成年', '性犯罪/未成年', '性犯罪/未成年', 
                 '暴力犯罪', '暴力犯罪', '暴力犯罪', '暴力犯罪'],
    '阶段': ['分类', '分类', '回归', '回归'] * 5,
    '评价指标': ['Cls_AUC', 'Cls_Recall', 'Final_R2', 'Final_MAE'] * 5,
    '完整特征模型 (Full Model)': [0.9132, 0.9192, 0.6461, 15.37, 
                                0.9120, 0.9072, 0.5530, 13.76, 
                                0.9128, 0.8921, 0.6186, 9.36, 
                                0.8409, 0.0340, 0.0391, 0.32, 
                                0.9012, 0.8928, 0.5264, 8.01],
    '剔除历史特征 (No History)': [0.8409, 0.8846, 0.3563, 23.88, 
                                0.8360, 0.8753, 0.1519, 21.05, 
                                0.8366, 0.8653, 0.2461, 15.06, 
                                0.7755, 0.0000, -0.0030, 0.36, 
                                0.8280, 0.8659, 0.2279, 10.82],
    '剔除微观街景 (No StreetView)': [0.8958, 0.9066, 0.6188, 16.37, 
                                  0.8970, 0.8923, 0.5531, 14.15, 
                                  0.8990, 0.8794, 0.6264, 9.42, 
                                  0.8432, 0.0291, -0.3935, 0.42, 
                                  0.8886, 0.8805, 0.4958, 8.58]
}
df_ny = pd.DataFrame(ny_data)

# 4. 写入 Excel (分两页)
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_hk.to_excel(writer, sheet_name='香港-交通冲突预测', index=False)
    df_ny.to_excel(writer, sheet_name='纽约-犯罪预测消融实验', index=False)

print(f"表格已成功导出至桌面：{output_path}")

In [ ]:
import os
import shap
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 0. 全局字体配置
# =========================================================
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# =========================================================
# 1. 文件夹配置 (改在当前 Jupyter 工作路径下，避开桌面权限)
# =========================================================
current_dir = os.getcwd()
hk_output_dir = os.path.join(current_dir, "HK_XGBoost_Final_Plots")
os.makedirs(hk_output_dir, exist_ok=True)
print(f"📁 图像保存目录已改至当前路径: {hk_output_dir}")

# =========================================================
# 2. 数据加载与对齐
# =========================================================
print("================ 1. 加载香港数据集 ================")
hk_file = r"E:\chuli\HK_cleaned_dataset_coarse.csv"
df = pd.read_csv(hk_file)

# 构建目标变量与历史先验
df['Y_Target'] = df['Year_2018_Freq'] + df['Year_2019_Freq']
df['X_Prior'] = (df['Year_2014_Freq'] + df['Year_2015_Freq'] + 
                 df['Year_2016_Freq'] + df['Year_2017_Freq'])

# 提取特征集
env_features = [col for col in df.columns if any(k in col for k in ['ratio_', 'accessibility', 'density', 'road'])]
history_feature = ['X_Prior']
full_features = env_features + history_feature

# =========================================================
# 3. 纯英文学术名称映射字典
# =========================================================
feature_name_mapping = {
    'HK_road_raster_clipped': 'Road Network Density',
    'ratio_HK_plus_Traffic_flow_clipped': 'Traffic Flow',
    'ratio_HK_plus_Enclosure_degree_clipped': 'Enclosure Degree',
    'ratio_HK_plus_Street_furniture_clipped': 'Street Furniture',
    'ratio_HK_plus_Pedestrian_density_clipped': 'Pedestrian Density',
    'HK_school_bus_accessibility_clipped': 'School Bus Accessibility',
    'ratio_HK_plus_Greening_degree_clipped': 'Greening Degree',
    'HK_school_to_shop_accessibility_10m_clipped': 'Shop Accessibility',
    'ratio_HK_plus_Openness_clipped': 'Sky Openness',
    'ratio_HK_plus_Protection_clipped': 'Physical Protection',
    'ratio_HK_plus_Surveillance_and_lighting_clipped': 'Lighting & Surveillance',
    'HK_bus_accessibility_clipped': 'Public Transit Accessibility',
    'X_Prior': 'Historical Risk Baseline' 
}

def get_academic_name(col_name):
    for key, val in feature_name_mapping.items():
        if key in col_name or col_name in key:
            return val
    clean_name = col_name.replace('ratio_HK_plus_', '').replace('ratio_HK_', '').replace('HK_', '')
    return clean_name.replace('_clipped', '').replace('_', ' ').title()

# =========================================================
# 4. 训练 XGBoost 分类器与计算 SHAP 
# =========================================================
print("🚀 正在训练 XGBoost 模型并计算 SHAP...")
X_full = df[full_features]
y = df['Y_Target']
X_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)

y_tr_cls = (y_train > 0).astype(int)

clf_xgb = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1)
clf_xgb.fit(X_train_full, y_tr_cls)

X_sample = X_train_full.sample(n=min(3000, len(X_train_full)), random_state=42)
explainer = shap.TreeExplainer(clf_xgb)
shap_vals = explainer.shap_values(X_sample)

if isinstance(shap_vals, list):
    shap_vals_target = shap_vals[1]
elif len(shap_vals.shape) == 3:
    shap_vals_target = shap_vals[:, :, 1] 
else:
    shap_vals_target = shap_vals 

# =========================================================
# 5. 剔除霸屏特征
# =========================================================
blacklist_features = ['X_Prior', 'POINT_X', 'POINT_Y'] 
cols_to_drop_idx = [X_sample.columns.get_loc(col) for col in blacklist_features if col in X_sample.columns]

if cols_to_drop_idx:
    shap_vals_filtered = np.delete(shap_vals_target, cols_to_drop_idx, axis=1)
    X_sample_filtered = X_sample.drop(columns=[col for col in blacklist_features if col in X_sample.columns])
else:
    shap_vals_filtered = shap_vals_target
    X_sample_filtered = X_sample.copy()

X_sample_renamed = X_sample_filtered.rename(columns=lambda x: get_academic_name(x))

# =========================================================
# 6. 【任务一】生成纯净版蜂群图 (Beeswarm Plot)
# =========================================================
print("\n📊 正在生成第一张图：XGBoost 纯净版蜂群图...")
plt.figure(figsize=(10.5, 6.5))
shap.summary_plot(shap_vals_filtered, X_sample_renamed, max_display=10, show=False)
plt.gcf().set_size_inches(11, 7)
plt.tight_layout()

beeswarm_path = os.path.join(hk_output_dir, "Figure_Z_SHAP_HK_XGBoost_Beeswarm.png")
plt.savefig(beeswarm_path, dpi=300, bbox_inches='tight')
print(f"--> 图 Z 已保存至硬盘: {os.path.exists(beeswarm_path)}")
plt.show() # 直接在网页端显示图片！

# =========================================================
# 7. 【任务二】抓取 Top 4 微观特征并生成 2x2 散点拼图
# =========================================================
print("\n🧩 正在提取 Top 4 微观特征并生成 2x2 拼图...")
mean_abs_shap = np.abs(shap_vals_filtered).mean(axis=0)
feat_importance = pd.DataFrame({
    'feature': X_sample_filtered.columns,
    'importance': mean_abs_shap
})

top_4_features_raw = feat_importance.sort_values(by='importance', ascending=False).head(4)['feature'].tolist()

temp_img_paths = []
for rank, raw_feat_name in enumerate(top_4_features_raw):
    fig, ax = plt.subplots(figsize=(8, 6))
    feat_academic = get_academic_name(raw_feat_name)
    try:
        shap.dependence_plot(
            feat_academic, 
            shap_vals_filtered, 
            X_sample_renamed, 
            interaction_index=None, 
            ax=ax,
            show=False,
            cmap=plt.get_cmap("cool") 
        )
        plt.title(f"XGBoost - HK Traffic Conflict\nTop {rank+1} Micro-Env Driver: {feat_academic}", fontsize=14, pad=15, fontweight='bold')
        plt.tight_layout()
        
        img_path = os.path.join(hk_output_dir, f"temp_scatter_{rank}.png")
        plt.savefig(img_path, dpi=300, bbox_inches='tight')
        temp_img_paths.append(img_path)
        plt.close(fig) # 隐藏单张小图，只显示最后的拼图
    except Exception as e:
        print(f"⚠️ 生成 {raw_feat_name} 散点图出错: {e}")

if len(temp_img_paths) == 4:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    labels = ['(a)', '(b)', '(c)', '(d)']
    for ax, img_path, label in zip(axes.flatten(), temp_img_paths, labels):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        ax.text(0.02, 0.98, label, transform=ax.transAxes, fontsize=24, fontweight='bold', va='top')
    
    plt.tight_layout()
    combined_path = os.path.join(hk_output_dir, "Figure_W_HK_Micro_Mechanisms_Combined.png")
    plt.savefig(combined_path, dpi=300, bbox_inches='tight')
    print(f"--> 图 W (拼图) 已保存至硬盘: {os.path.exists(combined_path)}")
    plt.show() # 直接在网页端显示拼图！
    
    # 清理单张图
    for p in temp_img_paths:
        os.remove(p)

print(f"\n================ 🎉 任务彻底完成！ ================")
print(f"如果由于系统权限导致硬盘里还是没有文件，请直接在上方显示的图片上【右键 -> 另存为】！")